# 🤖 Machine Learning: Box Office Revenue & Net ROI Predictor

This notebook implements a supervised Machine Learning regression model to predict worldwide box office revenue ($ Millions) and evaluate key drivers of return on investment (ROI) using Random Forest and Gradient Boosting Regressors.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

## 1. Data Ingestion & Feature Engineering

In [2]:
# Load budget data
tn = pd.read_csv('Data/tn.movie_budgets.csv')
tn['production_budget'] = tn['production_budget'].str.replace('$', '').str.replace(',', '').astype(float)
tn['worldwide_gross'] = tn['worldwide_gross'].str.replace('$', '').str.replace(',', '').astype(float)
tn['release_date'] = pd.to_datetime(tn['release_date'])
tn['release_month'] = tn['release_date'].dt.month
tn['release_year'] = tn['release_date'].dt.year

# Merge with TMDB ratings & popularity dataset
tmdb = pd.read_csv('Data/tmdb.movies.csv')
merged = pd.merge(tn, tmdb, left_on='movie', right_on='title', how='inner').drop_duplicates(subset=['movie', 'release_year'])
merged = merged[merged['production_budget'] > 0].dropna(subset=['worldwide_gross', 'popularity', 'vote_average', 'vote_count'])

# One-Hot Encode Top Genres
genre_map = {
    28: 'Action', 12: 'Adventure', 16: 'Animation', 35: 'Comedy', 80: 'Crime',
    99: 'Documentary', 18: 'Drama', 10751: 'Family', 14: 'Fantasy', 36: 'History',
    27: 'Horror', 10402: 'Music', 9648: 'Mystery', 10749: 'Romance', 878: 'Sci-Fi',
    53: 'Thriller', 10752: 'War', 37: 'Western'
}
for g_id, g_name in genre_map.items():
    merged[f'genre_{g_name}'] = merged['genre_ids'].apply(lambda x: 1 if str(g_id) in str(x) else 0)

## 2. Train Supervised Regression Model

In [3]:
feature_cols = ['production_budget', 'release_month', 'popularity', 'vote_average', 'vote_count'] + [f'genre_{g_name}' for g_name in genre_map.values()]
X = merged[feature_cols]
y = merged['worldwide_gross'] / 1e6

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
print(f'Random Forest Regressor R^2 Score: {r2_score(y_test, y_pred):.4f}')
print(f'Mean Absolute Error (MAE): ${mean_absolute_error(y_test, y_pred):.2f}M')
print(f'Root Mean Squared Error (RMSE): ${np.sqrt(mean_squared_error(y_test, y_pred)):.2f}M')

Random Forest Regressor R^2 Score: 0.7386
Mean Absolute Error (MAE): $52.20M
Root Mean Squared Error (RMSE): $103.34M


## 3. Feature Importance & Model Evaluation Plots

In [4]:
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False).head(10)
plt.figure(figsize=(10, 5))
sns.barplot(x=importances.values, y=importances.index, hue=importances.index, legend=False, palette='crest')
plt.title('Top 10 Feature Importances in Box Office Revenue Prediction', fontsize=13, fontweight='bold')
plt.xlabel('Relative Feature Importance Score')
plt.tight_layout()
plt.show()